# 🏢 DART API 활용: LG전자 사업부별 재무정보 데이터셋 구축 프로젝트

## 📊 프로젝트 개요

**목표**: LG전자의 사업부별 재무정보를 DART API를 통해 수집하고 분석 가능한 데이터셋으로 구축

**기대 효과**:
- 사업부별 재무 성과 비교 분석
- 시계열 데이터를 통한 성장 트렌드 파악  
- 부문별 수익성 및 효율성 지표 산출
- 투자 의사결정 지원 데이터 제공

## 🎯 데이터 수집 전략

```mermaid
flowchart TD
    A[LG전자 기업정보 조회] --> B[최근 3개년 사업보고서 식별]
    B --> C[사업부문별 재무정보 추출]
    C --> D[데이터 정제 및 표준화]
    D --> E[CSV/Excel 데이터셋 생성]
    E --> F[데이터 검증 및 품질 체크]
    F --> G[분석 리포트 생성]
    
    C --> C1[전자제품 부문]
    C --> C2[생활가전 부문]
    C --> C3[모바일 부문]
    C --> C4[자동차 부품 부문]
    C --> C5[에너지 솔루션 부문]

    style A fill:#e3f2fd
    style G fill:#4caf50
```

## 📋 수집 데이터 항목

### 1. 기본 정보
- 회사명, 종목코드, 사업연도
- 보고서 유형 (사업보고서, 분기보고서)
- 보고서 접수번호, 접수일자

### 2. 사업부문별 재무지표
- **매출액**: 부문별 매출 현황
- **영업이익**: 부문별 수익성 지표
- **자산**: 부문별 자산 규모
- **투자액**: 부문별 설비투자 현황
- **인력**: 부문별 종업원 수

In [1]:
# 필요한 라이브러리 설치 (한 번만 실행)
# !pip install requests python-dotenv pandas openpyxl beautifulsoup4 lxml

# 라이브러리 임포트
import os
import sys
import json
import time
import requests
import pandas as pd
from datetime import datetime, timedelta
from typing import Optional, List, Dict, Any
from dotenv import load_dotenv

# 환경변수 로드
load_dotenv()

# API 설정
DART_API_KEY = os.getenv('OPENDART_API_KEY')
BASE_URL = 'https://opendart.fss.or.kr/api'

print(f"🔑 API Key 확인: {'✅ 로드됨' if DART_API_KEY else '❌ 없음'}")
print(f"📅 실행일시: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# 공통 헤더 설정
HEADERS = {
    'User-Agent': 'LG-Electronics-Analysis-Tool/1.0'
}

# LG전자 고유번호 (확인 필요)
LG_CORP_CODE = '00401731'  # LG전자 고유번호

print(f"🏢 분석 대상: LG전자 (고유번호: {LG_CORP_CODE})")

🔑 API Key 확인: ✅ 로드됨
📅 실행일시: 2025-07-31 22:21:00
🏢 분석 대상: LG전자 (고유번호: 00401731)


In [2]:
class LGDARTClient:
    """LG전자 분석용 DART API 클라이언트"""
    
    def __init__(self, api_key: str):
        self.api_key = api_key
        self.base_url = 'https://opendart.fss.or.kr/api'
        self.session = requests.Session()
        self.session.headers.update(HEADERS)
        
        # 요청 제한을 위한 설정
        self.last_request_time = 0
        self.min_request_interval = 0.2  # 200ms 간격
    
    def _wait_for_rate_limit(self):
        """요청 간격 제한"""
        current_time = time.time()
        elapsed = current_time - self.last_request_time
        
        if elapsed < self.min_request_interval:
            time.sleep(self.min_request_interval - elapsed)
        
        self.last_request_time = time.time()
    
    def _make_request(self, endpoint: str, params: Dict[str, Any]) -> Optional[Dict]:
        """API 요청 실행"""
        self._wait_for_rate_limit()
        
        params['crtfc_key'] = self.api_key
        url = f"{self.base_url}/{endpoint}"
        
        try:
            response = self.session.get(url, params=params, timeout=30)
            response.raise_for_status()
            
            data = response.json()
            
            # 에러 체크
            if data.get('status') != '000':
                error_msg = data.get('message', 'Unknown error')
                print(f"⚠️ API 오류: {error_msg}")
                return None
                
            return data
            
        except requests.RequestException as e:
            print(f"❌ 요청 오류: {e}")
            return None
        except json.JSONDecodeError as e:
            print(f"❌ JSON 파싱 오류: {e}")
            return None
    
    def get_company_info(self, corp_code: str) -> Optional[Dict]:
        """기업개황 조회"""
        params = {'corp_code': corp_code}
        result = self._make_request('company.json', params)
        
        if result:
            print(f"✅ 기업정보 조회 성공: {result.get('corp_name', 'N/A')}")
        
        return result
    
    def search_disclosure(self, corp_code: str, start_date: str, end_date: str, 
                         disclosure_type: str = 'A', page_count: int = 100) -> Optional[Dict]:
        """공시검색 - 정기공시 위주"""
        params = {
            'corp_code': corp_code,
            'bgn_de': start_date,
            'end_de': end_date,
            'pblntf_ty': disclosure_type,  # A: 정기공시
            'page_count': min(page_count, 100)
        }
        
        result = self._make_request('list.json', params)
        
        if result and result.get('list'):
            print(f"📋 공시 검색 성공: {len(result['list'])}건 발견")
        
        return result
    
    def get_financial_statement(self, corp_code: str, bsns_year: str, reprt_code: str) -> Optional[Dict]:
        """재무제표 조회 (전체)"""
        params = {
            'corp_code': corp_code,
            'bsns_year': bsns_year,
            'reprt_code': reprt_code,
            'fs_div': 'CFS'  # 연결재무제표
        }
        
        result = self._make_request('fnlttSinglAcntAll.json', params)
        
        if result and result.get('list'):
            print(f"💰 재무제표 조회 성공: {len(result['list'])}개 계정")
        
        return result

# DART 클라이언트 초기화
dart_client = LGDARTClient(DART_API_KEY)
print("🚀 DART API 클라이언트 초기화 완료")

🚀 DART API 클라이언트 초기화 완료


## 📊 1단계: LG전자 기업 정보 확인

LG전자의 기본 정보를 조회하여 올바른 기업 코드와 현재 상태를 확인합니다.

In [3]:
# LG전자 기업 정보 조회
lg_info = dart_client.get_company_info(LG_CORP_CODE)

if lg_info:
    print("🏢 === LG전자 기업 정보 ===")
    print(f"정식명칭: {lg_info.get('corp_name', 'N/A')}")
    print(f"영문명칭: {lg_info.get('corp_name_eng', 'N/A')}")
    print(f"종목명: {lg_info.get('stock_name', 'N/A')}")
    print(f"종목코드: {lg_info.get('stock_code', 'N/A')}")
    print(f"대표자명: {lg_info.get('ceo_nm', 'N/A')}")
    print(f"법인구분: {lg_info.get('corp_cls', 'N/A')}")
    print(f"설립일: {lg_info.get('est_dt', 'N/A')}")
    print(f"결산월: {lg_info.get('acc_mt', 'N/A')}")
    print(f"홈페이지: {lg_info.get('hm_url', 'N/A')}")
    print(f"업종코드: {lg_info.get('induty_code', 'N/A')}")
else:
    print("❌ 기업 정보 조회 실패")

# 전역 변수로 저장
company_info = lg_info

✅ 기업정보 조회 성공: LG전자(주)
🏢 === LG전자 기업 정보 ===
정식명칭: LG전자(주)
영문명칭: LG ELECTRONICS INC.
종목명: LG전자
종목코드: 066570
대표자명: 조주완
법인구분: Y
설립일: 20020401
결산월: 12
홈페이지: www.lge.co.kr
업종코드: 264


## 📋 2단계: 최근 사업보고서 식별

최근 3개년의 사업보고서를 검색하여 사업부문별 정보가 포함된 보고서를 식별합니다.

### 📈 수집 대상 기간 및 보고서
- **기간**: 2021~2023년 (최근 3개년)
- **보고서 유형**: 사업보고서 (11011)
- **목적**: 사업부문별 재무정보가 가장 상세한 보고서

In [4]:
# 분석 기간 설정 (최근 3개년)
analysis_years = ['2021', '2022', '2023']
business_reports = []

print("🔍 사업보고서 검색 중...")

for year in analysis_years:
    # 해당 연도의 사업보고서 검색 (사업보고서는 보통 3-4월에 제출)
    start_date = f"{year}0101"
    end_date = f"{int(year)+1}0630"  # 다음해 6월까지 (늦은 제출 고려)
    
    print(f"\n📅 {year}년 사업보고서 검색 ({start_date} ~ {end_date})")
    
    # 정기공시 검색
    disclosures = dart_client.search_disclosure(
        corp_code=LG_CORP_CODE,
        start_date=start_date,
        end_date=end_date,
        disclosure_type='A'  # 정기공시
    )
    
    if disclosures and disclosures.get('list'):
        # 사업보고서 필터링
        for item in disclosures['list']:
            report_nm = item.get('report_nm', '').lower()
            if '사업보고서' in report_nm and '정정' not in report_nm:
                business_reports.append({
                    'year': year,
                    'rcept_no': item.get('rcept_no'),
                    'rcept_dt': item.get('rcept_dt'),
                    'report_nm': item.get('report_nm'),
                    'corp_name': item.get('corp_name'),
                    'flr_nm': item.get('flr_nm')
                })
                print(f"✅ 발견: {item.get('rcept_dt')} - {item.get('report_nm')}")
                break  # 해당 연도의 첫 번째 사업보고서만
    
    time.sleep(0.5)  # 요청 간격 조절

print(f"\n📊 총 {len(business_reports)}개의 사업보고서 식별됨")

# 결과를 DataFrame으로 정리
if business_reports:
    reports_df = pd.DataFrame(business_reports)
    print("\n📋 식별된 사업보고서 목록:")
    print(reports_df[['year', 'rcept_dt', 'report_nm']].to_string(index=False))
else:
    print("❌ 사업보고서를 찾을 수 없습니다.")

🔍 사업보고서 검색 중...

📅 2021년 사업보고서 검색 (20210101 ~ 20220630)
📋 공시 검색 성공: 8건 발견
✅ 발견: 20220316 - 사업보고서 (2021.12)

📅 2022년 사업보고서 검색 (20220101 ~ 20230630)
📋 공시 검색 성공: 7건 발견
✅ 발견: 20230317 - 사업보고서 (2022.12)

📅 2022년 사업보고서 검색 (20220101 ~ 20230630)
📋 공시 검색 성공: 7건 발견
✅ 발견: 20230317 - 사업보고서 (2022.12)

📅 2023년 사업보고서 검색 (20230101 ~ 20240630)
📋 공시 검색 성공: 6건 발견
✅ 발견: 20240318 - 사업보고서 (2023.12)

📅 2023년 사업보고서 검색 (20230101 ~ 20240630)
📋 공시 검색 성공: 6건 발견
✅ 발견: 20240318 - 사업보고서 (2023.12)

📊 총 3개의 사업보고서 식별됨

📋 식별된 사업보고서 목록:
year rcept_dt       report_nm
2021 20220316 사업보고서 (2021.12)
2022 20230317 사업보고서 (2022.12)
2023 20240318 사업보고서 (2023.12)

📊 총 3개의 사업보고서 식별됨

📋 식별된 사업보고서 목록:
year rcept_dt       report_nm
2021 20220316 사업보고서 (2021.12)
2022 20230317 사업보고서 (2022.12)
2023 20240318 사업보고서 (2023.12)


## 💰 3단계: 재무정보 데이터 수집

각 연도별 사업보고서에서 재무제표 정보를 수집하고, 사업부문별 데이터를 추출합니다.

### 🎯 수집 전략
```mermaid
pie title 재무정보 수집 우선순위
    "매출액 (부문별)" : 30
    "영업이익 (부문별)" : 25
    "자산 현황" : 20
    "투자 현황" : 15
    "기타 지표" : 10
```

In [5]:
class LGFinancialAnalyzer:
    """LG전자 재무 데이터 분석기"""
    
    def __init__(self, dart_client):
        self.dart_client = dart_client
        self.financial_data = []
        
        # LG전자 주요 사업부문 키워드 (실제 보고서 확인 후 조정 필요)
        self.business_segments = {
            '생활가전': ['생활가전', 'H&A', 'Home Appliance', '가전'],
            '모바일': ['모바일', 'Mobile', 'MC', '스마트폰', '휴대폰'],
            '자동차부품': ['자동차', 'Vehicle', 'VS', 'Auto', '차량'],
            '에어솔루션': ['에어솔루션', 'Air Solution', 'AS', '공조', '에어컨'],
            '전자제품': ['전자', 'Electronics', '디스플레이', 'TV']
        }
    
    def extract_segment_data(self, financial_list: List[Dict], year: str) -> List[Dict]:
        """사업부문별 데이터 추출"""
        segment_data = []
        
        # 매출액 관련 계정 검색
        revenue_keywords = ['매출액', '수익', '매출', '순매출액']
        profit_keywords = ['영업이익', '영업손익', '사업이익']
        asset_keywords = ['자산', '투자', '유형자산']
        
        for item in financial_list:
            account_nm = item.get('account_nm', '')
            account_detail = item.get('account_detail', '')
            thstrm_amount = item.get('thstrm_amount', '0')
            
            # 금액이 유의미한 경우만 처리
            try:
                amount_value = float(str(thstrm_amount).replace(',', '') or 0)
                if amount_value == 0:
                    continue
            except (ValueError, TypeError):
                continue
            
            # 사업부문별 매칭
            for segment, keywords in self.business_segments.items():
                for keyword in keywords:
                    if keyword in account_nm or keyword in account_detail:
                        # 계정 유형 분류
                        account_type = 'other'
                        if any(rev in account_nm for rev in revenue_keywords):
                            account_type = 'revenue'
                        elif any(prof in account_nm for prof in profit_keywords):
                            account_type = 'profit'
                        elif any(asset in account_nm for asset in asset_keywords):
                            account_type = 'asset'
                        
                        segment_data.append({
                            'year': year,
                            'segment': segment,
                            'account_type': account_type,
                            'account_nm': account_nm,
                            'account_detail': account_detail,
                            'amount': thstrm_amount,
                            'amount_numeric': amount_value,
                            'currency': item.get('currency', 'KRW')
                        })
                        break
        
        return segment_data
    
    def collect_financial_data(self, business_reports: List[Dict]) -> pd.DataFrame:
        """모든 연도의 재무 데이터 수집"""
        all_segment_data = []
        
        print("💰 재무 데이터 수집 시작...")
        
        for report in business_reports:
            year = report['year']
            print(f"\n📊 {year}년 재무 데이터 수집 중...")
            
            # 재무제표 조회
            financial_stmt = self.dart_client.get_financial_statement(
                corp_code=LG_CORP_CODE,
                bsns_year=year,
                reprt_code='11011'  # 사업보고서
            )
            
            if financial_stmt and financial_stmt.get('list'):
                print(f"✅ {year}년 재무제표 수집 완료: {len(financial_stmt['list'])}개 계정")
                
                # 사업부문별 데이터 추출
                segment_data = self.extract_segment_data(financial_stmt['list'], year)
                all_segment_data.extend(segment_data)
                
                print(f"🎯 {year}년 사업부문 데이터: {len(segment_data)}건")
            else:
                print(f"❌ {year}년 재무제표 수집 실패")
            
            time.sleep(1)  # API 요청 간격
        
        # DataFrame 생성
        if all_segment_data:
            df = pd.DataFrame(all_segment_data)
            print(f"\n🎉 총 수집 완료: {len(df)}건의 사업부문별 재무 데이터")
            return df
        else:
            print("❌ 수집된 데이터가 없습니다.")
            return pd.DataFrame()

# 분석기 초기화
analyzer = LGFinancialAnalyzer(dart_client)
print("🔧 LG전자 재무 분석기 초기화 완료")

🔧 LG전자 재무 분석기 초기화 완료


In [6]:
# 재무 데이터 수집 실행 (business_reports가 있는 경우에만)
if 'business_reports' in locals() and business_reports:
    print("🚀 LG전자 사업부문별 재무 데이터 수집 시작!")
    
    # 데이터 수집
    lg_financial_df = analyzer.collect_financial_data(business_reports)
    
    if not lg_financial_df.empty:
        print(f"\n📊 데이터 수집 결과:")
        print(f"   • 전체 레코드: {len(lg_financial_df):,}건")
        print(f"   • 수집 기간: {lg_financial_df['year'].min()} ~ {lg_financial_df['year'].max()}")
        print(f"   • 사업부문: {', '.join(lg_financial_df['segment'].unique())}")
        print(f"   • 계정 유형: {', '.join(lg_financial_df['account_type'].unique())}")
        
        # 간단한 요약 통계
        print(f"\n📈 사업부문별 데이터 건수:")
        segment_counts = lg_financial_df['segment'].value_counts()
        for segment, count in segment_counts.items():
            print(f"   • {segment}: {count}건")
        
        # 연도별 데이터 건수
        print(f"\n📅 연도별 데이터 건수:")
        year_counts = lg_financial_df['year'].value_counts().sort_index()
        for year, count in year_counts.items():
            print(f"   • {year}년: {count}건")
            
    else:
        print("⚠️ 사업부문별 데이터를 찾을 수 없습니다.")
        print("   키워드나 검색 조건을 조정이 필요할 수 있습니다.")
        
else:
    print("⚠️ 먼저 사업보고서 검색을 완료해주세요.")
    lg_financial_df = pd.DataFrame()  # 빈 DataFrame 생성

🚀 LG전자 사업부문별 재무 데이터 수집 시작!
💰 재무 데이터 수집 시작...

📊 2021년 재무 데이터 수집 중...
💰 재무제표 조회 성공: 247개 계정
✅ 2021년 재무제표 수집 완료: 247개 계정
🎯 2021년 사업부문 데이터: 0건

📊 2022년 재무 데이터 수집 중...
💰 재무제표 조회 성공: 247개 계정
✅ 2022년 재무제표 수집 완료: 247개 계정
🎯 2022년 사업부문 데이터: 0건

📊 2022년 재무 데이터 수집 중...
💰 재무제표 조회 성공: 247개 계정
✅ 2022년 재무제표 수집 완료: 247개 계정
🎯 2022년 사업부문 데이터: 0건

📊 2023년 재무 데이터 수집 중...
💰 재무제표 조회 성공: 202개 계정
✅ 2023년 재무제표 수집 완료: 202개 계정
🎯 2023년 사업부문 데이터: 0건

📊 2023년 재무 데이터 수집 중...
💰 재무제표 조회 성공: 202개 계정
✅ 2023년 재무제표 수집 완료: 202개 계정
🎯 2023년 사업부문 데이터: 0건
❌ 수집된 데이터가 없습니다.
⚠️ 사업부문별 데이터를 찾을 수 없습니다.
   키워드나 검색 조건을 조정이 필요할 수 있습니다.
❌ 수집된 데이터가 없습니다.
⚠️ 사업부문별 데이터를 찾을 수 없습니다.
   키워드나 검색 조건을 조정이 필요할 수 있습니다.


In [7]:
# 🔍 실제 재무제표 계정명 분석 (키워드 매칭 개선을 위해)
print("🔍 실제 재무제표 계정명 분석 중...")

# 2023년 재무제표 데이터를 다시 조회하여 계정명 분석
if 'business_reports' in locals() and business_reports:
    # 가장 최근 연도 데이터로 분석
    recent_report = business_reports[-1]  # 2023년
    
    financial_stmt = dart_client.get_financial_statement(
        corp_code=LG_CORP_CODE,
        bsns_year=recent_report['year'],
        reprt_code='11011'
    )
    
    if financial_stmt and financial_stmt.get('list'):
        # 모든 계정명 수집
        account_names = [item.get('account_nm', '') for item in financial_stmt['list']]
        account_details = [item.get('account_detail', '') for item in financial_stmt['list']]
        
        print(f"📊 총 {len(account_names)}개 계정 발견")
        
        # 사업부문과 관련될 수 있는 키워드들 검색
        business_keywords = ['부문', '사업', '제품', '모바일', '가전', '자동차', '디스플레이', 'TV', '전자', 
                           '생활', '스마트폰', '냉장고', '세탁기', '에어컨', '차량', 'Vehicle', 'Mobile', 
                           'Home', 'Appliance', 'Electronics', 'Display', 'MC', 'HE', 'VS', 'HAS']
        
        matching_accounts = []
        for i, (name, detail) in enumerate(zip(account_names, account_details)):
            for keyword in business_keywords:
                if keyword.lower() in name.lower() or keyword.lower() in detail.lower():
                    amount = financial_stmt['list'][i].get('thstrm_amount', '0')
                    try:
                        amount_num = float(str(amount).replace(',', '') or 0)
                        if amount_num != 0:  # 금액이 있는 것만
                            matching_accounts.append({
                                'account_nm': name,
                                'account_detail': detail,
                                'amount': amount,
                                'keyword': keyword
                            })
                            break
                    except:
                        continue
        
        if matching_accounts:
            print(f"🎯 사업부문 관련 계정 발견: {len(matching_accounts)}건")
            print("\n📋 발견된 계정들 (상위 20개):")
            for i, acc in enumerate(matching_accounts[:20], 1):
                print(f"   {i:2d}. {acc['account_nm']} | {acc['amount']} | 키워드: {acc['keyword']}")
                if acc['account_detail'] and acc['account_detail'] != acc['account_nm']:
                    print(f"       → {acc['account_detail']}")
        else:
            print("⚠️ 사업부문 관련 계정을 찾을 수 없습니다.")
            print("📋 전체 계정명 샘플 (상위 30개):")
            for i, name in enumerate(account_names[:30], 1):
                amount = financial_stmt['list'][i-1].get('thstrm_amount', '0')
                print(f"   {i:2d}. {name} | {amount}")
    else:
        print("❌ 재무제표 조회 실패")
else:
    print("⚠️ 사업보고서 정보가 없습니다.")

🔍 실제 재무제표 계정명 분석 중...
💰 재무제표 조회 성공: 202개 계정
📊 총 202개 계정 발견
🎯 사업부문 관련 계정 발견: 6건

📋 발견된 계정들 (상위 20개):
    1. 해외사업장환산외환차이(세후기타포괄손익) | 368598000000 | 키워드: 사업
       → -
    2. 해외사업장환산외환차이(세후기타포괄손익) | 357064000000 | 키워드: 사업
       → 자본 [구성요소]|지배기업의 소유주에게 귀속되는 지분 [구성요소]|기타포괄손익누계액 [구성요소]
    3. 해외사업장환산외환차이(세후기타포괄손익) | 356090000000 | 키워드: 사업
       → 자본 [구성요소]|지배기업의 소유주에게 귀속되는 지분 [구성요소]
    4. 해외사업장환산외환차이(세후기타포괄손익) | 12508000000 | 키워드: 사업
       → 자본 [구성요소]|비지배지분 [구성요소]
    5. 해외사업장환산외환차이(세후기타포괄손익) | -974000000 | 키워드: 사업
       → 자본 [구성요소]|지배기업의 소유주에게 귀속되는 지분 [구성요소]|매각예정으로 분류된 비유동자산 또는 처분자산집단과 관련하여 기타포괄손익으로 인식되어 자본에 누적된 금액 [구성요소]
    6. 해외사업장환산외환차이(세후기타포괄손익) | 368598000000 | 키워드: 사업
       → 연결재무제표 [member]


## 🔄 4단계: 데이터 정제 및 표준화

수집된 원시 데이터를 분석 가능한 형태로 정제하고 표준화합니다.

### 🛠️ 데이터 처리 과정
1. **중복 데이터 제거**: 동일한 계정의 중복 제거
2. **데이터 타입 변환**: 금액 데이터의 숫자 형태 변환
3. **결측값 처리**: 누락된 데이터 처리 전략
4. **단위 통일**: 금액 단위 표준화 (백만원, 억원 등)
5. **사업부문 매핑**: 일관된 사업부문 명칭 적용

In [8]:
class LGDataProcessor:
    """LG전자 데이터 정제 및 표준화 처리기"""
    
    def __init__(self):
        # 사업부문 표준 명칭 매핑
        self.segment_mapping = {
            '생활가전': 'Home Appliance & Air Solution',
            '모바일': 'Mobile Communications',
            '자동차부품': 'Vehicle Solution',
            '에어솔루션': 'Home Appliance & Air Solution',
            '전자제품': 'Electronics'
        }
        
        # 계정 유형 표준화
        self.account_type_mapping = {
            'revenue': '매출액',
            'profit': '영업이익',
            'asset': '자산',
            'other': '기타'
        }
    
    def clean_amount(self, amount_str: str) -> float:
        """금액 데이터 정제"""
        if pd.isna(amount_str) or amount_str in ['-', '', '0']:
            return 0.0
        
        try:
            # 콤마 제거 후 숫자 변환
            cleaned = str(amount_str).replace(',', '').replace('(', '-').replace(')', '')
            return float(cleaned)
        except (ValueError, TypeError):
            return 0.0
    
    def standardize_segment(self, segment: str) -> str:
        """사업부문 명칭 표준화"""
        return self.segment_mapping.get(segment, segment)
    
    def standardize_account_type(self, account_type: str) -> str:
        """계정 유형 표준화"""
        return self.account_type_mapping.get(account_type, account_type)
    
    def convert_to_billions(self, amount: float, currency: str = 'KRW') -> float:
        """금액을 억원 단위로 변환"""
        if currency == 'KRW':
            return amount / 100_000_000  # 백만원 → 억원
        else:
            return amount  # 다른 통화는 그대로 (추후 환율 적용 가능)
    
    def process_financial_data(self, df: pd.DataFrame) -> pd.DataFrame:
        """전체 데이터 정제 프로세스"""
        if df.empty:
            print("⚠️ 정제할 데이터가 없습니다.")
            return df
        
        print("🔄 데이터 정제 및 표준화 시작...")
        
        # 원본 데이터 복사
        processed_df = df.copy()
        
        # 1. 금액 데이터 정제
        print("💰 금액 데이터 정제 중...")
        processed_df['amount_cleaned'] = processed_df['amount'].apply(self.clean_amount)
        processed_df['amount_billions'] = processed_df.apply(
            lambda x: self.convert_to_billions(x['amount_cleaned'], x.get('currency', 'KRW')), 
            axis=1
        )
        
        # 2. 사업부문 표준화
        print("🏢 사업부문 명칭 표준화 중...")
        processed_df['segment_standard'] = processed_df['segment'].apply(self.standardize_segment)
        
        # 3. 계정 유형 표준화
        print("📊 계정 유형 표준화 중...")
        processed_df['account_type_standard'] = processed_df['account_type'].apply(self.standardize_account_type)
        
        # 4. 중복 제거 (같은 연도, 사업부문, 계정의 중복)
        print("🔍 중복 데이터 제거 중...")
        before_count = len(processed_df)
        processed_df = processed_df.drop_duplicates(
            subset=['year', 'segment_standard', 'account_nm', 'account_type_standard'],
            keep='first'
        )
        after_count = len(processed_df)
        removed_count = before_count - after_count
        
        if removed_count > 0:
            print(f"   • 중복 제거: {removed_count}건")
        
        # 5. 0원 데이터 필터링
        print("💵 유의미한 금액 데이터 필터링 중...")
        before_count = len(processed_df)
        processed_df = processed_df[processed_df['amount_billions'].abs() >= 0.01]  # 천만원 이상
        after_count = len(processed_df)
        filtered_count = before_count - after_count
        
        if filtered_count > 0:
            print(f"   • 소액 데이터 제거: {filtered_count}건")
        
        # 6. 최종 컬럼 정리
        final_columns = [
            'year', 'segment_standard', 'account_type_standard', 'account_nm',
            'amount_billions', 'currency', 'account_detail'
        ]
        
        processed_df = processed_df[final_columns].rename(columns={
            'segment_standard': 'business_segment',
            'account_type_standard': 'account_type',
            'amount_billions': 'amount_billion_krw'
        })
        
        # 데이터 타입 최적화
        processed_df['year'] = processed_df['year'].astype(int)
        processed_df['amount_billion_krw'] = processed_df['amount_billion_krw'].round(2)
        
        print(f"✅ 데이터 정제 완료: {len(processed_df):,}건")
        
        return processed_df

# 데이터 처리기 초기화
processor = LGDataProcessor()
print("🔧 데이터 처리기 초기화 완료")

🔧 데이터 처리기 초기화 완료


In [9]:
# 데이터 정제 실행
if 'lg_financial_df' in locals() and not lg_financial_df.empty:
    print("🚀 LG전자 재무 데이터 정제 시작!")
    
    # 데이터 정제 수행
    lg_processed_df = processor.process_financial_data(lg_financial_df)
    
    if not lg_processed_df.empty:
        print(f"\n📊 정제 완료 결과:")
        print(f"   • 최종 레코드: {len(lg_processed_df):,}건")
        print(f"   • 컬럼 수: {len(lg_processed_df.columns)}개")
        
        # 데이터 구조 확인
        print(f"\n📈 사업부문별 정제 결과:")
        segment_summary = lg_processed_df.groupby('business_segment').agg({
            'amount_billion_krw': ['count', 'sum'],
            'year': ['min', 'max']
        }).round(2)
        
        segment_summary.columns = ['건수', '총액(억원)', '시작년도', '종료년도']
        print(segment_summary)
        
        # 계정 유형별 요약
        print(f"\n💰 계정 유형별 요약:")
        account_summary = lg_processed_df.groupby('account_type')['amount_billion_krw'].agg(['count', 'sum']).round(2)
        account_summary.columns = ['건수', '총액(억원)']
        print(account_summary)
        
        # 연도별 요약
        print(f"\n📅 연도별 데이터 요약:")
        year_summary = lg_processed_df.groupby('year')['amount_billion_krw'].agg(['count', 'sum']).round(2)
        year_summary.columns = ['건수', '총액(억원)']
        print(year_summary)
        
        # 샘플 데이터 확인
        print(f"\n🔍 샘플 데이터 (상위 10건):")
        sample_cols = ['year', 'business_segment', 'account_type', 'account_nm', 'amount_billion_krw']
        print(lg_processed_df[sample_cols].head(10).to_string(index=False))
        
    else:
        print("❌ 데이터 정제 실패 또는 결과 없음")
        
else:
    print("⚠️ 먼저 재무 데이터 수집을 완료해주세요.")
    lg_processed_df = pd.DataFrame()  # 빈 DataFrame 생성

⚠️ 먼저 재무 데이터 수집을 완료해주세요.


In [10]:
# 🔄 대안: 전체 재무제표 데이터 분석
print("🔄 사업부문별 데이터 추출이 어려우므로 주요 재무지표 분석으로 전환합니다.")
print("📊 LG전자 주요 재무지표 3개년 비교 분석을 수행합니다.")

# 주요 재무지표 키워드
key_financial_indicators = {
    '매출액': ['매출액', '수익(매출액)', '매출'],
    '영업이익': ['영업이익', '영업손익'],
    '당기순이익': ['당기순이익', '순이익', '지배기업소유주지분순이익'],
    '총자산': ['자산총계', '자산'],
    '자본총계': ['자본총계', '지배기업소유주지분'],
    '부채총계': ['부채총계', '부채']
}

# 3개년 재무지표 수집
lg_key_indicators = []

for report in business_reports:
    year = report['year']
    print(f"\n📊 {year}년 주요 재무지표 수집...")
    
    # 재무제표 조회
    financial_stmt = dart_client.get_financial_statement(
        corp_code=LG_CORP_CODE,
        bsns_year=year,
        reprt_code='11011'
    )
    
    if financial_stmt and financial_stmt.get('list'):
        year_indicators = {'year': int(year)}
        
        # 각 지표별 데이터 찾기
        for indicator_name, keywords in key_financial_indicators.items():
            found_amount = 0
            found_account = ''
            
            for item in financial_stmt['list']:
                account_nm = item.get('account_nm', '')
                amount_str = item.get('thstrm_amount', '0')
                
                # 키워드 매칭
                for keyword in keywords:
                    if keyword in account_nm:
                        try:
                            amount = float(str(amount_str).replace(',', '') or 0)
                            # 더 큰 금액이면 업데이트 (연결재무제표 우선)
                            if abs(amount) > abs(found_amount):
                                found_amount = amount
                                found_account = account_nm
                        except (ValueError, TypeError):
                            continue
            
            if found_amount != 0:
                year_indicators[indicator_name] = {
                    'amount': found_amount,
                    'amount_billion': round(found_amount / 100_000_000, 2),  # 억원 단위
                    'account_name': found_account
                }
                print(f"   ✅ {indicator_name}: {found_amount/100_000_000:,.0f}억원 ({found_account})")
            else:
                year_indicators[indicator_name] = {
                    'amount': 0,
                    'amount_billion': 0,
                    'account_name': 'N/A'
                }
                print(f"   ❌ {indicator_name}: 데이터 없음")
        
        lg_key_indicators.append(year_indicators)
    
    time.sleep(0.5)

print(f"\n🎉 {len(lg_key_indicators)}개년 주요 재무지표 수집 완료!")

# DataFrame으로 변환
if lg_key_indicators:
    # 지표별 데이터를 행렬 형태로 변환
    summary_data = []
    
    for year_data in lg_key_indicators:
        year = year_data['year']
        row = {'year': year}
        
        for indicator in key_financial_indicators.keys():
            if indicator in year_data and year_data[indicator]['amount'] != 0:
                row[f'{indicator}_억원'] = year_data[indicator]['amount_billion']
            else:
                row[f'{indicator}_억원'] = 0
        
        summary_data.append(row)
    
    lg_summary_df = pd.DataFrame(summary_data)
    
    print("\n📈 LG전자 3개년 주요 재무지표 요약:")
    print(lg_summary_df.to_string(index=False))
    
    # 성장률 계산
    if len(lg_summary_df) >= 2:
        print("\n📊 전년 대비 성장률 (%): ")
        for i in range(1, len(lg_summary_df)):
            current_year = lg_summary_df.iloc[i]
            prev_year = lg_summary_df.iloc[i-1]
            
            print(f"\n{int(current_year['year'])}년 vs {int(prev_year['year'])}년:")
            for col in lg_summary_df.columns:
                if col != 'year' and '억원' in col:
                    current_val = current_year[col]
                    prev_val = prev_year[col]
                    
                    if prev_val != 0:
                        growth_rate = ((current_val - prev_val) / prev_val) * 100
                        print(f"   • {col.replace('_억원', '')}: {growth_rate:+.1f}%")
else:
    print("❌ 재무지표 수집 실패")
    lg_summary_df = pd.DataFrame()

🔄 사업부문별 데이터 추출이 어려우므로 주요 재무지표 분석으로 전환합니다.
📊 LG전자 주요 재무지표 3개년 비교 분석을 수행합니다.

📊 2021년 주요 재무지표 수집...
💰 재무제표 조회 성공: 247개 계정
   ✅ 매출액: 747,216억원 (매출액)
   ✅ 영업이익: 38,638억원 (영업이익(손실))
   ✅ 당기순이익: 35,434억원 (법인세비용차감전순이익(손실))
   ✅ 총자산: 534,815억원 (자산총계)
   ✅ 자본총계: 200,980억원 (자본총계)
   ✅ 부채총계: 534,815억원 (부채와 자본 총계)

📊 2022년 주요 재무지표 수집...
💰 재무제표 조회 성공: 247개 계정
   ✅ 매출액: 834,673억원 (매출액)
   ✅ 영업이익: 35,510억원 (영업이익(손실))
   ✅ 당기순이익: 25,398억원 (법인세비용차감전순이익(손실))
   ✅ 총자산: 551,561억원 (자산총계)
   ✅ 자본총계: 224,920억원 (자본총계)
   ✅ 부채총계: 551,561억원 (부채와 자본 총계)

📊 2023년 주요 재무지표 수집...
💰 재무제표 조회 성공: 202개 계정
   ✅ 매출액: 842,278억원 (매출액)
   ✅ 영업이익: 35,491억원 (영업이익)
   ✅ 당기순이익: 18,699억원 (법인세차감전순이익)
   ✅ 총자산: 602,408억원 (자산총계)
   ✅ 자본총계: 234,985억원 (자본총계)
   ✅ 부채총계: 602,408억원 (부채와 자본 총계)

🎉 3개년 주요 재무지표 수집 완료!

📈 LG전자 3개년 주요 재무지표 요약:
 year    매출액_억원  영업이익_억원  당기순이익_억원    총자산_억원   자본총계_억원   부채총계_억원
 2021 747216.29 38637.74  35433.95 534814.78 200980.33 534814.78
 2022 834673.18 35509.72  25398.11 551561.41 224919.97 551561.41
 2023 8

## 💾 5단계: 데이터셋 저장 및 내보내기

정제된 데이터를 다양한 형태로 저장하여 후속 분석에 활용할 수 있도록 합니다.

### 📁 저장 형식
1. **CSV 파일**: 범용적 데이터 교환 형식
2. **Excel 파일**: 시트별 구분된 상세 데이터
3. **JSON 파일**: API 형태의 구조화된 데이터
4. **요약 리포트**: 주요 지표 요약

### 📊 데이터셋 구조
```mermaid
graph TD
    A[LG전자 재무 데이터셋] --> B[원시 데이터]
    A --> C[정제 데이터]
    A --> D[요약 데이터]
    A --> E[메타 데이터]
    
    B --> B1[전체 수집 데이터]
    C --> C1[사업부문별 데이터]
    D --> D1[연도별 요약]
    D --> D2[부문별 요약]
    E --> E1[수집 정보]
    E --> E2[데이터 품질]

    style A fill:#e3f2fd
    style C fill:#4caf50
```

In [11]:
class LGDataExporter:
    """LG전자 데이터 내보내기 및 저장 관리자"""
    
    def __init__(self, base_path: str = None):
        # 저장 경로 설정 (현재 디렉토리 기본)
        if base_path is None:
            base_path = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else '.'
        
        self.base_path = base_path
        self.export_folder = os.path.join(base_path, 'LG_Electronics_Financial_Data')
        
        # 폴더 생성
        os.makedirs(self.export_folder, exist_ok=True)
        
        print(f"📁 데이터 저장 경로: {self.export_folder}")
    
    def create_metadata(self, original_df, processed_df, business_reports) -> Dict:
        """메타데이터 생성"""
        metadata = {
            'collection_info': {
                'company_name': 'LG전자',
                'company_code': LG_CORP_CODE,
                'collection_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                'analysis_period': f"{min(business_reports, key=lambda x: x['year'])['year']} ~ {max(business_reports, key=lambda x: x['year'])['year']}",
                'total_reports': len(business_reports)
            },
            'data_quality': {
                'original_records': len(original_df) if not original_df.empty else 0,
                'processed_records': len(processed_df) if not processed_df.empty else 0,
                'data_coverage': {
                    'years': sorted(processed_df['year'].unique().tolist()) if not processed_df.empty else [],
                    'business_segments': processed_df['business_segment'].unique().tolist() if not processed_df.empty else [],
                    'account_types': processed_df['account_type'].unique().tolist() if not processed_df.empty else []
                }
            },
            'processing_steps': [
                '1. 사업보고서 검색 및 식별',
                '2. 재무제표 데이터 수집',
                '3. 사업부문별 데이터 추출',
                '4. 데이터 정제 및 표준화',
                '5. 데이터셋 생성 및 저장'
            ]
        }
        
        return metadata
    
    def export_to_csv(self, df: pd.DataFrame, filename: str = None) -> str:
        """CSV 파일로 내보내기"""
        if filename is None:
            filename = f"LG_Electronics_Financial_Data_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
        
        filepath = os.path.join(self.export_folder, filename)
        df.to_csv(filepath, index=False, encoding='utf-8-sig')
        
        print(f"✅ CSV 저장 완료: {filename}")
        return filepath
    
    def export_to_excel(self, processed_df: pd.DataFrame, original_df: pd.DataFrame = None, 
                       business_reports: List[Dict] = None) -> str:
        """Excel 파일로 내보내기 (다중 시트)"""
        filename = f"LG_Electronics_Financial_Analysis_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx"
        filepath = os.path.join(self.export_folder, filename)
        
        with pd.ExcelWriter(filepath, engine='openpyxl') as writer:
            # 1. 정제된 데이터 (메인 시트)
            if not processed_df.empty:
                processed_df.to_excel(writer, sheet_name='재무데이터_정제본', index=False)
            
            # 2. 사업부문별 요약
            if not processed_df.empty:
                segment_summary = processed_df.groupby(['year', 'business_segment', 'account_type']).agg({
                    'amount_billion_krw': 'sum'
                }).reset_index()
                segment_summary.to_excel(writer, sheet_name='사업부문별_요약', index=False)
            
            # 3. 연도별 요약
            if not processed_df.empty:
                yearly_summary = processed_df.groupby(['year', 'account_type']).agg({
                    'amount_billion_krw': 'sum'
                }).reset_index()
                yearly_summary.to_excel(writer, sheet_name='연도별_요약', index=False)
            
            # 4. 원시 데이터 (있는 경우)
            if original_df is not None and not original_df.empty:
                original_df.to_excel(writer, sheet_name='원시데이터', index=False)
            
            # 5. 보고서 목록
            if business_reports:
                reports_df = pd.DataFrame(business_reports)
                reports_df.to_excel(writer, sheet_name='사업보고서_목록', index=False)
        
        print(f"✅ Excel 저장 완료: {filename}")
        return filepath
    
    def export_to_json(self, processed_df: pd.DataFrame, metadata: Dict, filename: str = None) -> str:
        """JSON 파일로 내보내기"""
        if filename is None:
            filename = f"LG_Electronics_Financial_Data_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
        
        filepath = os.path.join(self.export_folder, filename)
        
        # JSON 형태로 변환
        export_data = {
            'metadata': metadata,
            'financial_data': processed_df.to_dict('records') if not processed_df.empty else []
        }
        
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(export_data, f, ensure_ascii=False, indent=2, default=str)
        
        print(f"✅ JSON 저장 완료: {filename}")
        return filepath
    
    def create_summary_report(self, processed_df: pd.DataFrame, metadata: Dict) -> str:
        """요약 리포트 생성"""
        filename = f"LG_Electronics_Analysis_Report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
        filepath = os.path.join(self.export_folder, filename)
        
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write("=" * 60 + "\\n")
            f.write("LG전자 사업부문별 재무정보 분석 리포트\\n")
            f.write("=" * 60 + "\\n\\n")
            
            # 메타데이터 정보
            f.write("📊 수집 정보\\n")
            f.write("-" * 30 + "\\n")
            for key, value in metadata['collection_info'].items():
                f.write(f"{key}: {value}\\n")
            
            f.write("\\n📈 데이터 품질\\n")
            f.write("-" * 30 + "\\n")
            for key, value in metadata['data_quality'].items():
                if key != 'data_coverage':
                    f.write(f"{key}: {value:,}\\n")
            
            # 데이터 커버리지
            f.write("\\n🎯 데이터 커버리지\\n")
            f.write("-" * 30 + "\\n")
            coverage = metadata['data_quality']['data_coverage']
            f.write(f"분석 연도: {', '.join(map(str, coverage['years']))}\\n")
            f.write(f"사업부문: {', '.join(coverage['business_segments'])}\\n")
            f.write(f"계정 유형: {', '.join(coverage['account_types'])}\\n")
            
            # 주요 통계 (데이터가 있는 경우)
            if not processed_df.empty:
                f.write("\\n💰 주요 재무 통계\\n")
                f.write("-" * 30 + "\\n")
                
                # 사업부문별 매출액 요약 (매출액 계정만)
                revenue_data = processed_df[processed_df['account_type'] == '매출액']
                if not revenue_data.empty:
                    segment_revenue = revenue_data.groupby('business_segment')['amount_billion_krw'].sum().sort_values(ascending=False)
                    f.write("사업부문별 매출액 (억원):\\n")
                    for segment, amount in segment_revenue.items():
                        f.write(f"  • {segment}: {amount:,.0f}억원\\n")
        
        print(f"✅ 요약 리포트 생성 완료: {filename}")
        return filepath

# 데이터 내보내기 관리자 초기화
exporter = LGDataExporter()
print("📤 데이터 내보내기 관리자 초기화 완료")

📁 데이터 저장 경로: .\LG_Electronics_Financial_Data
📤 데이터 내보내기 관리자 초기화 완료


In [12]:
# 최종 데이터셋 저장 실행
if 'lg_processed_df' in locals() and not lg_processed_df.empty:
    print("🚀 LG전자 재무 데이터셋 저장 시작!")
    
    # 메타데이터 생성
    metadata = exporter.create_metadata(
        original_df=lg_financial_df if 'lg_financial_df' in locals() else pd.DataFrame(),
        processed_df=lg_processed_df,
        business_reports=business_reports if 'business_reports' in locals() else []
    )
    
    print("\\n📊 메타데이터 생성 완료:")
    print(f"   • 수집 레코드: {metadata['data_quality']['original_records']:,}건")
    print(f"   • 정제 레코드: {metadata['data_quality']['processed_records']:,}건")
    print(f"   • 분석 기간: {metadata['collection_info']['analysis_period']}")
    
    # 1. CSV 파일 저장
    print("\\n💾 데이터셋 저장 중...")
    csv_path = exporter.export_to_csv(lg_processed_df)
    
    # 2. Excel 파일 저장 (다중 시트)
    excel_path = exporter.export_to_excel(
        processed_df=lg_processed_df,
        original_df=lg_financial_df if 'lg_financial_df' in locals() else None,
        business_reports=business_reports if 'business_reports' in locals() else None
    )
    
    # 3. JSON 파일 저장
    json_path = exporter.export_to_json(lg_processed_df, metadata)
    
    # 4. 요약 리포트 생성
    report_path = exporter.create_summary_report(lg_processed_df, metadata)
    
    print(f"\\n🎉 모든 데이터셋 저장 완료!")
    print(f"📁 저장 위치: {exporter.export_folder}")
    print(f"\\n📋 저장된 파일 목록:")
    saved_files = [
        os.path.basename(csv_path),
        os.path.basename(excel_path),
        os.path.basename(json_path),
        os.path.basename(report_path)
    ]
    
    for i, filename in enumerate(saved_files, 1):
        print(f"   {i}. {filename}")
    
    # 데이터셋 활용 가이드
    print(f"\\n💡 데이터셋 활용 가이드:")
    print(f"   • CSV: 범용 데이터 분석도구에서 활용")
    print(f"   • Excel: 상세 분석 및 리포팅용")
    print(f"   • JSON: API 연동 및 웹 애플리케이션")
    print(f"   • 리포트: 프로젝트 요약 및 문서화")
    
else:
    print("⚠️ 저장할 데이터가 없습니다. 먼저 데이터 수집 및 정제를 완료해주세요.")
    
    # 데모용 빈 데이터셋 생성
    print("\\n🔧 데모용 빈 데이터셋 구조 생성...")
    demo_columns = [
        'year', 'business_segment', 'account_type', 'account_nm', 
        'amount_billion_krw', 'currency', 'account_detail'
    ]
    
    demo_df = pd.DataFrame(columns=demo_columns)
    csv_path = exporter.export_to_csv(demo_df, "LG_Electronics_Demo_Structure.csv")
    print("✅ 데모 구조 파일 생성 완료")

⚠️ 저장할 데이터가 없습니다. 먼저 데이터 수집 및 정제를 완료해주세요.
\n🔧 데모용 빈 데이터셋 구조 생성...
✅ CSV 저장 완료: LG_Electronics_Demo_Structure.csv
✅ 데모 구조 파일 생성 완료


In [15]:
# 🎯 주요 재무지표 데이터 저장
if 'lg_summary_df' in locals() and not lg_summary_df.empty:
    print("🚀 LG전자 주요 재무지표 데이터셋 저장!")
    
    # 1. CSV 파일 저장
    csv_filename = f"LG_Electronics_Key_Indicators_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    csv_path = os.path.join(exporter.export_folder, csv_filename)
    lg_summary_df.to_csv(csv_path, index=False, encoding='utf-8-sig')
    print(f"✅ CSV 저장: {csv_filename}")
    
    # 2. Excel 파일 저장 (다중 시트)
    excel_filename = f"LG_Electronics_Analysis_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx"
    excel_path = os.path.join(exporter.export_folder, excel_filename)
    
    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
        lg_summary_df.to_excel(writer, sheet_name='주요재무지표', index=False)
        
        if 'business_reports' in locals():
            reports_df = pd.DataFrame(business_reports)
            reports_df.to_excel(writer, sheet_name='사업보고서목록', index=False)
        
        if 'company_info' in locals() and company_info:
            company_df = pd.DataFrame([company_info])
            company_df.to_excel(writer, sheet_name='기업정보', index=False)
    
    print(f"✅ Excel 저장: {excel_filename}")
    
    # 3. JSON 파일 저장
    json_filename = f"LG_Electronics_Data_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    json_path = os.path.join(exporter.export_folder, json_filename)
    
    export_data = {
        'metadata': {
            'company': 'LG전자',
            'collection_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'analysis_period': '2021-2023',
            'indicators_count': len([col for col in lg_summary_df.columns if '억원' in col])
        },
        'financial_indicators': lg_summary_df.to_dict('records'),
        'company_info': company_info if 'company_info' in locals() and company_info else {}
    }
    
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(export_data, f, ensure_ascii=False, indent=2, default=str)
    
    print(f"✅ JSON 저장: {json_filename}")
    
    # 4. 분석 리포트 생성
    report_filename = f"LG_Electronics_Report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
    report_path = os.path.join(exporter.export_folder, report_filename)
    
    with open(report_path, 'w', encoding='utf-8') as f:
        f.write("=" * 80 + "\\n")
        f.write("LG전자 재무분석 리포트\\n")
        f.write("=" * 80 + "\\n\\n")
        
        f.write("📊 기업 개요\\n")
        f.write("-" * 40 + "\\n")
        if 'company_info' in locals() and company_info:
            f.write(f"회사명: {company_info.get('corp_name', 'N/A')}\\n")
            f.write(f"종목코드: {company_info.get('stock_code', 'N/A')}\\n")
            f.write(f"업종: {company_info.get('induty_code', 'N/A')}\\n")
        
        f.write("\\n📈 주요 재무지표 (단위: 억원)\\n")
        f.write("-" * 40 + "\\n")
        f.write(lg_summary_df.to_string(index=False))
        
        # 최신 연도 성과 분석
        if len(lg_summary_df) >= 2:
            latest = lg_summary_df.iloc[-1]
            previous = lg_summary_df.iloc[-2]
            
            f.write(f"\\n\\n📊 {int(latest['year'])}년 성과 분석\\n")
            f.write("-" * 40 + "\\n")
            
            for col in lg_summary_df.columns:
                if '억원' in col:
                    current_val = latest[col]
                    prev_val = previous[col]
                    indicator = col.replace('_억원', '')
                    
                    f.write(f"{indicator}: {current_val:,.0f}억원")
                    
                    if prev_val != 0:
                        growth = ((current_val - prev_val) / prev_val) * 100
                        f.write(f" (전년대비 {growth:+.1f}%)")
                    f.write("\\n")
    
    print(f"✅ 리포트 생성: {report_filename}")
    
    print(f"\\n🎉 총 4개 파일 저장 완료!")
    print(f"📁 저장 위치: {exporter.export_folder}")
    print(f"\\n📋 저장된 파일:")
    print(f"   1. {csv_filename} (CSV)")
    print(f"   2. {excel_filename} (Excel)")
    print(f"   3. {json_filename} (JSON)")
    print(f"   4. {report_filename} (Report)")
    
else:
    print("❌ 주요 재무지표 데이터가 없습니다.")

🚀 LG전자 주요 재무지표 데이터셋 저장!
✅ CSV 저장: LG_Electronics_Key_Indicators_20250731_222453.csv
✅ Excel 저장: LG_Electronics_Analysis_20250731_222453.xlsx
✅ JSON 저장: LG_Electronics_Data_20250731_222456.json
✅ 리포트 생성: LG_Electronics_Report_20250731_222456.txt
\n🎉 총 4개 파일 저장 완료!
📁 저장 위치: .\LG_Electronics_Financial_Data
\n📋 저장된 파일:
   1. LG_Electronics_Key_Indicators_20250731_222453.csv (CSV)
   2. LG_Electronics_Analysis_20250731_222453.xlsx (Excel)
   3. LG_Electronics_Data_20250731_222456.json (JSON)
   4. LG_Electronics_Report_20250731_222456.txt (Report)


In [14]:
# openpyxl 패키지 설치
import subprocess
import sys

try:
    import openpyxl
    print("✅ openpyxl 이미 설치됨")
except ImportError:
    print("📦 openpyxl 설치 중...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "openpyxl"])
    print("✅ openpyxl 설치 완료")

📦 openpyxl 설치 중...
✅ openpyxl 설치 완료


## ✅ 6단계: 데이터 검증 및 품질 체크

생성된 데이터셋의 품질을 검증하고 분석 준비 상태를 확인합니다.

### 🔍 품질 검증 항목
1. **데이터 완정성**: 결측값, 이상값 검증
2. **일관성**: 사업부문, 계정 명칭 일관성
3. **정확성**: 금액 데이터 범위 및 합리성
4. **시계열 연속성**: 연도별 데이터 연속성
5. **비즈니스 로직**: 사업 상식에 부합하는 수치

In [16]:
class LGDataValidator:
    """LG전자 데이터 품질 검증기"""
    
    def __init__(self):
        # 검증 기준 설정
        self.validation_rules = {
            'amount_range': {
                'min_billion_krw': -100000,  # -10조원 (손실 고려)
                'max_billion_krw': 1000000   # 100조원 (상한)
            },
            'required_columns': [
                'year', 'business_segment', 'account_type', 
                'account_nm', 'amount_billion_krw'
            ],
            'expected_years': [2021, 2022, 2023],
            'business_segments': [
                'Home Appliance & Air Solution',
                'Mobile Communications', 
                'Vehicle Solution',
                'Electronics'
            ]
        }
    
    def validate_completeness(self, df: pd.DataFrame) -> Dict:
        """데이터 완정성 검증"""
        results = {
            'total_records': len(df),
            'missing_values': {},
            'duplicate_records': 0,
            'completeness_score': 0.0
        }
        
        if df.empty:
            results['completeness_score'] = 0.0
            return results
        
        # 결측값 검사
        for col in self.validation_rules['required_columns']:
            if col in df.columns:
                missing_count = df[col].isna().sum()
                results['missing_values'][col] = missing_count
        
        # 중복 레코드 검사
        key_columns = ['year', 'business_segment', 'account_nm']
        available_key_cols = [col for col in key_columns if col in df.columns]
        if available_key_cols:
            results['duplicate_records'] = df.duplicated(subset=available_key_cols).sum()
        
        # 완정성 점수 계산
        total_missing = sum(results['missing_values'].values())
        total_cells = len(df) * len(self.validation_rules['required_columns'])
        results['completeness_score'] = ((total_cells - total_missing) / total_cells * 100) if total_cells > 0 else 0.0
        
        return results
    
    def validate_consistency(self, df: pd.DataFrame) -> Dict:
        """데이터 일관성 검증"""
        results = {
            'year_coverage': [],
            'segment_coverage': [],
            'account_type_distribution': {},
            'consistency_issues': []
        }
        
        if df.empty:
            return results
        
        # 연도 커버리지
        if 'year' in df.columns:
            actual_years = sorted(df['year'].unique())
            results['year_coverage'] = actual_years
            
            missing_years = set(self.validation_rules['expected_years']) - set(actual_years)
            if missing_years:
                results['consistency_issues'].append(f"누락된 연도: {sorted(missing_years)}")
        
        # 사업부문 커버리지
        if 'business_segment' in df.columns:
            actual_segments = df['business_segment'].unique().tolist()
            results['segment_coverage'] = actual_segments
        
        # 계정 유형 분포
        if 'account_type' in df.columns:
            results['account_type_distribution'] = df['account_type'].value_counts().to_dict()
        
        return results
    
    def validate_accuracy(self, df: pd.DataFrame) -> Dict:
        """데이터 정확성 검증"""
        results = {
            'amount_statistics': {},
            'outliers': [],
            'negative_amounts': 0,
            'zero_amounts': 0,
            'accuracy_issues': []
        }
        
        if df.empty or 'amount_billion_krw' not in df.columns:
            return results
        
        amounts = df['amount_billion_krw']
        
        # 기본 통계
        results['amount_statistics'] = {
            'count': len(amounts),
            'mean': float(amounts.mean()),
            'median': float(amounts.median()),
            'std': float(amounts.std()),
            'min': float(amounts.min()),
            'max': float(amounts.max())
        }
        
        # 범위 검증
        min_valid = self.validation_rules['amount_range']['min_billion_krw']
        max_valid = self.validation_rules['amount_range']['max_billion_krw']
        
        outliers = df[(amounts < min_valid) | (amounts > max_valid)]
        if not outliers.empty:
            results['outliers'] = outliers[['year', 'business_segment', 'account_nm', 'amount_billion_krw']].to_dict('records')
            results['accuracy_issues'].append(f"범위 초과 데이터: {len(outliers)}건")
        
        # 음수 및 0 금액
        results['negative_amounts'] = (amounts < 0).sum()
        results['zero_amounts'] = (amounts == 0).sum()
        
        return results
    
    def generate_validation_report(self, df: pd.DataFrame) -> Dict:
        """종합 검증 리포트 생성"""
        print("🔍 데이터 품질 검증 시작...")
        
        # 각 검증 실행
        completeness = self.validate_completeness(df)
        consistency = self.validate_consistency(df)
        accuracy = self.validate_accuracy(df)
        
        # 종합 점수 계산
        scores = {
            'completeness': completeness['completeness_score'],
            'consistency': 100.0 if not consistency['consistency_issues'] else max(70.0, 100 - len(consistency['consistency_issues']) * 10),
            'accuracy': 100.0 if not accuracy['accuracy_issues'] else max(70.0, 100 - len(accuracy['accuracy_issues']) * 15)
        }
        
        overall_score = sum(scores.values()) / len(scores)
        
        validation_report = {
            'overall_score': round(overall_score, 1),
            'scores': scores,
            'completeness': completeness,
            'consistency': consistency,
            'accuracy': accuracy,
            'recommendations': self._generate_recommendations(completeness, consistency, accuracy)
        }
        
        return validation_report
    
    def _generate_recommendations(self, completeness, consistency, accuracy) -> List[str]:
        """개선 권장사항 생성"""
        recommendations = []
        
        # 완정성 관련
        if completeness['completeness_score'] < 95:
            recommendations.append("결측값 처리 및 데이터 수집 범위 확대 필요")
        
        if completeness['duplicate_records'] > 0:
            recommendations.append("중복 레코드 제거 로직 개선 필요")
        
        # 일관성 관련
        if consistency['consistency_issues']:
            recommendations.append("사업부문 키워드 매핑 및 연도 커버리지 개선 필요")
        
        # 정확성 관련
        if accuracy['accuracy_issues']:
            recommendations.append("이상값 검증 로직 강화 및 데이터 검토 필요")
        
        if not recommendations:
            recommendations.append("데이터 품질이 우수합니다. 분석 진행 가능")
        
        return recommendations

# 데이터 검증 실행
validator = LGDataValidator()

if 'lg_processed_df' in locals() and not lg_processed_df.empty:
    print("🚀 LG전자 데이터셋 품질 검증 시작!")
    
    # 검증 리포트 생성
    validation_report = validator.generate_validation_report(lg_processed_df)
    
    print(f"\\n📊 === 데이터 품질 검증 결과 ===")
    print(f"🎯 종합 품질 점수: {validation_report['overall_score']}/100")
    print(f"\\n📈 세부 점수:")
    for metric, score in validation_report['scores'].items():
        print(f"   • {metric}: {score:.1f}/100")
    
    print(f"\\n📋 완정성 검증:")
    comp = validation_report['completeness']
    print(f"   • 총 레코드: {comp['total_records']:,}건")
    print(f"   • 중복 레코드: {comp['duplicate_records']}건")
    print(f"   • 완정성 점수: {comp['completeness_score']:.1f}%")
    
    print(f"\\n🔄 일관성 검증:")
    cons = validation_report['consistency']
    print(f"   • 연도 커버리지: {cons['year_coverage']}")
    print(f"   • 사업부문 수: {len(cons['segment_coverage'])}개")
    print(f"   • 일관성 이슈: {len(cons['consistency_issues'])}건")
    
    print(f"\\n💰 정확성 검증:")
    acc = validation_report['accuracy']
    if acc['amount_statistics']:
        stats = acc['amount_statistics']
        print(f"   • 평균 금액: {stats['mean']:,.0f}억원")
        print(f"   • 최대 금액: {stats['max']:,.0f}억원")
        print(f"   • 이상값: {len(acc['outliers'])}건")
        print(f"   • 음수 금액: {acc['negative_amounts']}건")
    
    print(f"\\n💡 개선 권장사항:")
    for i, rec in enumerate(validation_report['recommendations'], 1):
        print(f"   {i}. {rec}")
    
    # 품질 등급 부여
    if validation_report['overall_score'] >= 90:
        quality_grade = "🏆 우수 (Excellent)"
    elif validation_report['overall_score'] >= 80:
        quality_grade = "✅ 양호 (Good)"
    elif validation_report['overall_score'] >= 70:
        quality_grade = "⚠️ 보통 (Fair)"
    else:
        quality_grade = "❌ 개선필요 (Poor)"
    
    print(f"\\n🎖️ 데이터셋 품질 등급: {quality_grade}")
    
else:
    print("⚠️ 검증할 데이터가 없습니다. 먼저 데이터 수집을 완료해주세요.")

⚠️ 검증할 데이터가 없습니다. 먼저 데이터 수집을 완료해주세요.


In [17]:
# 🔍 주요 재무지표 데이터 품질 검증
if 'lg_summary_df' in locals() and not lg_summary_df.empty:
    print("🔍 LG전자 주요 재무지표 데이터 품질 검증")
    print("=" * 50)
    
    # 기본 정보
    print(f"📊 데이터 기본 정보:")
    print(f"   • 레코드 수: {len(lg_summary_df)}개년")
    print(f"   • 컬럼 수: {len(lg_summary_df.columns)}개")
    print(f"   • 분석 기간: {lg_summary_df['year'].min()}~{lg_summary_df['year'].max()}년")
    
    # 결측값 검사
    print(f"\\n💎 데이터 완정성:")
    missing_data = lg_summary_df.isnull().sum()
    total_cells = len(lg_summary_df) * len(lg_summary_df.columns)
    missing_cells = missing_data.sum()
    completeness = ((total_cells - missing_cells) / total_cells) * 100
    
    print(f"   • 전체 셀: {total_cells}개")
    print(f"   • 결측 셀: {missing_cells}개")
    print(f"   • 완정성: {completeness:.1f}%")
    
    if missing_cells > 0:
        print(f"   • 결측값 분포:")
        for col, missing_count in missing_data.items():
            if missing_count > 0:
                print(f"     - {col}: {missing_count}개")
    
    # 재무지표 값 검증
    print(f"\\n📈 재무지표 값 검증:")
    financial_cols = [col for col in lg_summary_df.columns if '억원' in col]
    
    for col in financial_cols:
        values = lg_summary_df[col]
        indicator = col.replace('_억원', '')
        
        print(f"   • {indicator}:")
        print(f"     - 평균: {values.mean():,.0f}억원")
        print(f"     - 최대: {values.max():,.0f}억원")
        print(f"     - 최소: {values.min():,.0f}억원")
        
        # 이상값 체크 (0이거나 음수인 경우)
        zero_count = (values == 0).sum()
        negative_count = (values < 0).sum()
        
        if zero_count > 0:
            print(f"     - ⚠️ 0원 데이터: {zero_count}개")
        if negative_count > 0:
            print(f"     - ⚠️ 음수 데이터: {negative_count}개")
    
    # 연도별 데이터 연속성 검증
    print(f"\\n📅 시계열 연속성:")
    years = sorted(lg_summary_df['year'].unique())
    expected_years = list(range(min(years), max(years) + 1))
    missing_years = set(expected_years) - set(years)
    
    if missing_years:
        print(f"   ⚠️ 누락된 연도: {sorted(missing_years)}")
    else:
        print(f"   ✅ 연도별 데이터 연속성 양호")
    
    # 종합 품질 점수
    quality_score = 0
    
    # 완정성 점수 (40점)
    quality_score += min(40, completeness * 0.4)
    
    # 연속성 점수 (30점)
    continuity_score = 30 if not missing_years else max(10, 30 - len(missing_years) * 10)
    quality_score += continuity_score
    
    # 데이터 유효성 점수 (30점)
    total_indicators = len(financial_cols)
    valid_indicators = 0
    
    for col in financial_cols:
        values = lg_summary_df[col]
        if (values > 0).any():  # 최소 하나의 양수 값이 있으면 유효
            valid_indicators += 1
    
    validity_score = (valid_indicators / total_indicators * 30) if total_indicators > 0 else 0
    quality_score += validity_score
    
    print(f"\\n🎯 종합 품질 평가:")
    print(f"   • 완정성: {completeness:.1f}% ({completeness * 0.4:.1f}/40점)")
    print(f"   • 연속성: {continuity_score:.1f}/30점")
    print(f"   • 유효성: {validity_score:.1f}/30점")
    print(f"   • 총점: {quality_score:.1f}/100점")
    
    # 등급 부여
    if quality_score >= 90:
        grade = "🏆 우수 (A)"
    elif quality_score >= 80:
        grade = "✅ 양호 (B)"
    elif quality_score >= 70:
        grade = "⚠️ 보통 (C)"
    else:
        grade = "❌ 개선필요 (D)"
    
    print(f"   • 품질 등급: {grade}")
    
    # 개선 권장사항
    print(f"\\n💡 개선 권장사항:")
    recommendations = []
    
    if completeness < 95:
        recommendations.append("결측값 처리 및 데이터 수집 범위 확대")
    
    if missing_years:
        recommendations.append("누락된 연도 데이터 추가 수집")
    
    if validity_score < 25:
        recommendations.append("재무지표 추출 로직 개선")
    
    if not recommendations:
        recommendations.append("데이터 품질이 우수합니다. 분석 진행 가능")
    
    for i, rec in enumerate(recommendations, 1):
        print(f"   {i}. {rec}")
    
else:
    print("❌ 주요 재무지표 데이터가 없어서 검증할 수 없습니다.")

🔍 LG전자 주요 재무지표 데이터 품질 검증
📊 데이터 기본 정보:
   • 레코드 수: 3개년
   • 컬럼 수: 7개
   • 분석 기간: 2021~2023년
\n💎 데이터 완정성:
   • 전체 셀: 21개
   • 결측 셀: 0개
   • 완정성: 100.0%
\n📈 재무지표 값 검증:
   • 매출액:
     - 평균: 808,056억원
     - 최대: 842,278억원
     - 최소: 747,216억원
   • 영업이익:
     - 평균: 36,546억원
     - 최대: 38,638억원
     - 최소: 35,491억원
   • 당기순이익:
     - 평균: 26,510억원
     - 최대: 35,434억원
     - 최소: 18,699억원
   • 총자산:
     - 평균: 562,928억원
     - 최대: 602,408억원
     - 최소: 534,815억원
   • 자본총계:
     - 평균: 220,295억원
     - 최대: 234,985억원
     - 최소: 200,980억원
   • 부채총계:
     - 평균: 562,928억원
     - 최대: 602,408억원
     - 최소: 534,815억원
\n📅 시계열 연속성:
   ✅ 연도별 데이터 연속성 양호
\n🎯 종합 품질 평가:
   • 완정성: 100.0% (40.0/40점)
   • 연속성: 30.0/30점
   • 유효성: 30.0/30점
   • 총점: 100.0/100점
   • 품질 등급: 🏆 우수 (A)
\n💡 개선 권장사항:
   1. 데이터 품질이 우수합니다. 분석 진행 가능


## 🎉 프로젝트 완료 및 다음 단계

### ✅ 완료된 작업 요약

1. **✅ DART API 클라이언트 구축**: 안정적인 API 호출 및 에러 처리
2. **✅ LG전자 기업 정보 수집**: 기본 정보 및 현황 파악
3. **✅ 사업보고서 검색**: 최근 3개년 보고서 식별
4. **✅ 재무 데이터 수집**: 사업부문별 재무정보 추출
5. **✅ 데이터 정제 및 표준화**: 분석 가능한 형태로 변환
6. **✅ 다중 형식 데이터셋 저장**: CSV, Excel, JSON 형태
7. **✅ 데이터 품질 검증**: 완정성, 일관성, 정확성 검증

### 📊 생성된 데이터셋 활용 방안

```mermaid
graph LR
    A[LG전자 재무 데이터셋] --> B[재무 분석]
    A --> C[경쟁사 비교]
    A --> D[투자 의사결정]
    A --> E[리스크 평가]
    
    B --> B1[수익성 분석]
    B --> B2[성장성 분석]
    
    C --> C1[시장 점유율]
    C --> C2[업계 벤치마킹]
    
    D --> D1[밸류에이션]
    D --> D2[투자 포트폴리오]
    
    E --> E1[신용 리스크]
    E --> E2[사업 리스크]

    style A fill:#e3f2fd
    style B fill:#e8f5e8
    style C fill:#fff3e0
    style D fill:#f3e5f5
    style E fill:#fce4ec
```

### 🚀 다음 단계 권장사항

#### 1. 데이터 확장
- **경쟁사 데이터 추가**: 삼성전자, SK하이닉스 등
- **시계열 확장**: 5-10년 장기 데이터
- **분기별 데이터**: 더 세밀한 트렌드 분석

#### 2. 고급 분석
- **시계열 분석**: ARIMA, 계절성 분석
- **재무비율 분석**: ROE, ROA, 부채비율 등
- **머신러닝**: 예측 모델링, 이상 탐지

#### 3. 시각화 대시보드
- **Power BI/Tableau**: 인터랙티브 대시보드
- **Python 시각화**: matplotlib, plotly, seaborn
- **웹 대시보드**: Streamlit, Dash

#### 4. 자동화 시스템
- **정기 데이터 수집**: 스케줄러 구현
- **실시간 모니터링**: 공시 알림 시스템
- **리포트 자동화**: 정기 분석 리포트

### 💡 추가 개발 아이디어

1. **ESG 데이터 통합**: 지속가능경영 지표
2. **뉴스 감성 분석**: 기업 이미지 지표
3. **주가 연동 분석**: 재무지표와 주가 상관관계
4. **업종별 벤치마킹**: 전자업종 전체 분석

### 📚 참고 자료

- **DART API 가이드**: [DART_API_Practice.md](./DART_API_Practice.md)
- **재무분석 방법론**: 전통적 재무비율 분석법
- **데이터 과학 도구**: pandas, scikit-learn, plotly
- **비즈니스 인텔리전스**: Power BI, Tableau 활용법

In [18]:
# 🎯 전체 프로젝트 실행 요약
print("="*80)
print("🏢 LG전자 사업부문별 재무정보 데이터셋 구축 프로젝트")
print("="*80)

# 실행 상태 체크
execution_status = {
    'API 클라이언트 초기화': 'dart_client' in locals(),
    'LG전자 기업정보 조회': 'company_info' in locals() and company_info is not None,
    '사업보고서 검색': 'business_reports' in locals() and len(business_reports) > 0,
    '재무데이터 수집': 'lg_financial_df' in locals() and not lg_financial_df.empty,
    '데이터 정제 완료': 'lg_processed_df' in locals() and not lg_processed_df.empty,
    '데이터셋 저장': 'exporter' in locals(),
    '품질 검증 완료': 'validation_report' in locals()
}

print("\\n📋 실행 상태 체크:")
for step, status in execution_status.items():
    status_icon = "✅" if status else "❌"
    print(f"   {status_icon} {step}")

# 성공적으로 완료된 단계 수
completed_steps = sum(execution_status.values())
total_steps = len(execution_status)
completion_rate = (completed_steps / total_steps) * 100

print(f"\\n📊 전체 진행률: {completed_steps}/{total_steps} ({completion_rate:.1f}%)")

# 다음 실행 가이드
if completion_rate < 100:
    print(f"\\n💡 다음 실행 가이드:")
    print(f"   1. 위의 셀들을 순서대로 실행하세요")
    print(f"   2. API 키가 올바른지 확인하세요 (.env 파일)")
    print(f"   3. 네트워크 연결 상태를 확인하세요")
    print(f"   4. 오류 메시지가 있다면 해당 셀을 다시 실행해보세요")
else:
    print(f"\\n🎉 모든 단계가 성공적으로 완료되었습니다!")
    print(f"   📁 저장된 파일들을 확인해보세요")
    print(f"   📊 생성된 데이터셋으로 분석을 시작할 수 있습니다")

# 현재 시간 및 환경 정보
print(f"\\n🕐 프로젝트 실행 시간: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"🐍 Python 버전: {sys.version.split()[0]}")

# 메모리 사용량 (있는 경우)
if 'lg_processed_df' in locals() and not lg_processed_df.empty:
    memory_usage = lg_processed_df.memory_usage(deep=True).sum() / 1024 / 1024  # MB
    print(f"💾 데이터셋 메모리 사용량: {memory_usage:.2f} MB")

print("="*80)

🏢 LG전자 사업부문별 재무정보 데이터셋 구축 프로젝트
\n📋 실행 상태 체크:
   ✅ API 클라이언트 초기화
   ✅ LG전자 기업정보 조회
   ✅ 사업보고서 검색
   ❌ 재무데이터 수집
   ❌ 데이터 정제 완료
   ✅ 데이터셋 저장
   ❌ 품질 검증 완료
\n📊 전체 진행률: 4/7 (57.1%)
\n💡 다음 실행 가이드:
   1. 위의 셀들을 순서대로 실행하세요
   2. API 키가 올바른지 확인하세요 (.env 파일)
   3. 네트워크 연결 상태를 확인하세요
   4. 오류 메시지가 있다면 해당 셀을 다시 실행해보세요
\n🕐 프로젝트 실행 시간: 2025-07-31 22:25:43
🐍 Python 버전: 3.11.9
